# **Modelo de Classificação com metodo Ensemble**

# `XGBoost para prever fraude em transações de criptomoedas`

Os dados foram extraídos do link abaixo:
>https://www.kaggle.com/datasets/vagifa/ethereum-frauddetection-dataset/data

In [1]:
# Imports
import numpy as np 
import pandas as pd 

import sklearn
import xgboost
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from joblib import dump, load
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings('ignore')

#### Carregando e Compreendendo os Dados

In [2]:
# Carregando os dados

df = pd.read_csv('dataset.csv')

In [3]:
# Shape
df.shape

(9841, 51)

In [4]:
# Visualizando uma amostra dos dados

df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


#### Variável Alvo

In [5]:
# Variável alvo

df.FLAG.value_counts()

FLAG
0    7662
1    2179
Name: count, dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 51 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            9841 non-null   int64  
 1   Index                                                 9841 non-null   int64  
 2   Address                                               9841 non-null   object 
 3   FLAG                                                  9841 non-null   int64  
 4   Avg min between sent tnx                              9841 non-null   float64
 5   Avg min between received tnx                          9841 non-null   float64
 6   Time Diff between first and last (Mins)               9841 non-null   float64
 7   Sent tnx                                              9841 non-null   int64  
 8   Received Tnx                                          9841

#### Engenharia de Atributos

In [7]:
# Visualizando os dados

df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [8]:
df.columns

Index(['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx',
       'Avg min between received tnx',
       'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx',
       'Number of Created Contracts', 'Unique Received From Addresses',
       'Unique Sent To Addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'min value sent to contract', 'max val sent to contract',
       'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' Total ERC20 tnxs', ' ERC20 total Ether received',
       ' ERC20 total ether sent', ' ERC20 total Ether sent contract',
       ' ERC20 uniq sent addr', ' ERC20 uniq rec addr',
       ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr',
       ' ERC20 avg time between sent tnx', ' ERC20 

**Algumas colunas estão com Letras maiusculas outras em minusculo e nós iremos padronizar para minusculo(lower) e vamos retirar os espaços em branco no inicio e final dos nomes**

In [9]:
# Ajusta o nome para minúsculo e retira o espaço no início do nome

# df.columns = [x.strip().lower() for x in df.columns]

df.columns = df.columns.map(str.strip).map(str.lower)

In [10]:
# Essas colunas não serão relevantes para análise então vamos remove-las

cols_to_drop = ['erc20 most sent token type',
                'erc20_most_rec_token_type',
                'address',
                'index',
                'unnamed: 0']

**Pegando todas as colunas do DataFrame (df.columns),** **`exceto`** aquelas que estão em **cols_to_drop** e também a coluna **'flag'**

In [11]:
# Selecionando os atributos filtrando as colunas que serão removidas e a variável alvo = cols_to_drop + ['flag']

atributos = [x for x in df.columns if x not in cols_to_drop + ['flag']]

In [12]:
atributos

['avg min between sent tnx',
 'avg min between received tnx',
 'time diff between first and last (mins)',
 'sent tnx',
 'received tnx',
 'number of created contracts',
 'unique received from addresses',
 'unique sent to addresses',
 'min value received',
 'max value received',
 'avg val received',
 'min val sent',
 'max val sent',
 'avg val sent',
 'min value sent to contract',
 'max val sent to contract',
 'avg value sent to contract',
 'total transactions (including tnx to create contract',
 'total ether sent',
 'total ether received',
 'total ether sent contracts',
 'total ether balance',
 'total erc20 tnxs',
 'erc20 total ether received',
 'erc20 total ether sent',
 'erc20 total ether sent contract',
 'erc20 uniq sent addr',
 'erc20 uniq rec addr',
 'erc20 uniq sent addr.1',
 'erc20 uniq rec contract addr',
 'erc20 avg time between sent tnx',
 'erc20 avg time between rec tnx',
 'erc20 avg time between rec 2 tnx',
 'erc20 avg time between contract tnx',
 'erc20 min val rec',
 'erc20

In [13]:
# Extraindo a quantidade de valores únicos

valores_unicos = df.nunique()

In [14]:
valores_unicos

unnamed: 0                                              9841
index                                                   4729
address                                                 9816
flag                                                       2
avg min between sent tnx                                5013
avg min between received tnx                            6223
time diff between first and last (mins)                 7810
sent tnx                                                 641
received tnx                                             727
number of created contracts                               20
unique received from addresses                           256
unique sent to addresses                                 258
min value received                                      4589
max value received                                      6302
avg val received                                        6767
min val sent                                            4719
max val sent            

In [15]:
# Mantém somente atributos com mais de um valor único (atributos que não são constantes)

atributos = [x for x in atributos if valores_unicos[x] > 1]

In [16]:
df[atributos].nunique()

avg min between sent tnx                                5013
avg min between received tnx                            6223
time diff between first and last (mins)                 7810
sent tnx                                                 641
received tnx                                             727
number of created contracts                               20
unique received from addresses                           256
unique sent to addresses                                 258
min value received                                      4589
max value received                                      6302
avg val received                                        6767
min val sent                                            4719
max val sent                                            6647
avg val sent                                            5854
min value sent to contract                                 3
max val sent to contract                                   4
avg value sent to contra

In [17]:
df[atributos].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 38 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   avg min between sent tnx                              9841 non-null   float64
 1   avg min between received tnx                          9841 non-null   float64
 2   time diff between first and last (mins)               9841 non-null   float64
 3   sent tnx                                              9841 non-null   int64  
 4   received tnx                                          9841 non-null   int64  
 5   number of created contracts                           9841 non-null   int64  
 6   unique received from addresses                        9841 non-null   int64  
 7   unique sent to addresses                              9841 non-null   int64  
 8   min value received                                    9841

In [18]:
# separar X e y

X = df[atributos]
y = df['flag']

In [19]:
# divisão treino e teste

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [20]:
# pipeline de preprocessamento

pipeline_numerico = Pipeline([('imputer', SimpleImputer(strategy='mean')),('scaler', StandardScaler())])

In [21]:
pipeline_numerico

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'mean'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


1️⃣ SimpleImputer(strategy='mean')

* `SimpleImputer()` ➡ preencher valores faltantes (NaN).

* `strategy='mean'` ➡ substituir valores faltantes pela média da coluna.

2️⃣ StandardScaler()

* padronizar os dados numéricos transformando os valores para terem:

        média = 0
        desvio padrão = 1

In [22]:
# column transformer

preprocessamento = ColumnTransformer([('num', pipeline_numerico, atributos)])

In [23]:
preprocessamento

,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


* `Função do ColumnTransformer()`: Dizer em quais colunas aplicar o pipeline.
* Ele pega o pipeline anterior e aplica nas colunas:

In [24]:
# pipeline final

pipeline_final = Pipeline([('preprocessamento', preprocessamento),
                           ('modelo', XGBClassifier(random_state=42, eval_metric='auc', objective='binary:logistic'))])

In [25]:
# treino

pipeline_final.fit(X_train, y_train)

,steps,"[('preprocessamento', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### 📊 Estrutura

>`Os três códigos formam camadas:`

    Pipeline final
    │
    ├── preprocessamento (ColumnTransformer)
    │        │
    │        └── pipeline_numerico
    │                │
    │                ├── SimpleImputer
    │                └── StandardScaler
    │
    └── modelo (XGBoost)

In [26]:
# previsão

predicoes = pipeline_final.predict(X_test)

In [27]:
print(classification_report(y_test, predicoes))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1533
           1       0.98      0.94      0.96       436

    accuracy                           0.98      1969
   macro avg       0.98      0.97      0.97      1969
weighted avg       0.98      0.98      0.98      1969



In [28]:
score_auc = metrics.roc_auc_score(y_test, predicoes)
print(f'AUC nos Dados de Teste - {score_auc:,.2%}')

AUC nos Dados de Teste - 96.61%


In [29]:
# ✅  Salvando o modelo

joblib.dump(pipeline_final, 'modelo_final.joblib')

['modelo_final.joblib']

In [30]:
# ✅  Carregar o modelo (deploy)

modelo_carregado = joblib.load('modelo_final.joblib')

#### Deploy do Modelo e Detecção de Fraudes em Novas Transações com Criptomoedas

In [31]:
# Fazendo previsões com novos dados

novos_dados = pd.read_csv('novos_dados.csv')

In [32]:
# Visualizando novos_dados

novos_dados

,avg min between sent tnx,avg min between received tnx,time diff between first and last (mins),sent tnx,received tnx,number of created contracts,unique received from addresses,unique sent to addresses,min value received,max value received,...,erc20 uniq sent addr.1,erc20 uniq rec contract addr,erc20 min val rec,erc20 max val rec,erc20 avg val rec,erc20 min val sent,erc20 max val sent,erc20 avg val sent,erc20 uniq sent token name,erc20 uniq rec token name
0,2570.59,3336.01,30572.7,8,3,0,2,4,0.1,40.0,...,0.0,1.0,600.0,600.0,600.0,0.0,0.0,0.0,0.0,1.0


In [33]:
# garantindo mesmo padrão de colunas

novos_dados.columns = novos_dados.columns.str.strip().str.lower()

In [34]:
# Previsão de maior probabilidade

previsao = modelo_carregado.predict(novos_dados)

In [35]:
previsao

array([0])

In [36]:
# Resultado

if previsao[0] == 0:
    print("Segundo o modelo, provavelmente, essa transação não representa uma Fraude.")
else:
    print("🚨 Segundo o modelo, provavelmente, essa transação pode representar uma Fraude. Acione verificação humana! 🚨")

Segundo o modelo, provavelmente, essa transação não representa uma Fraude.


### **Conclusão**

Este projeto demonstrou a construção de um sistema de detecção de fraudes utilizando técnicas modernas de Machine Learning e engenharia de dados.

A utilização de pipelines do Scikit-Learn garante um fluxo robusto, reprodutível e pronto para integração em sistemas reais.

A metodologia aplicada pode ser facilmente adaptada para diversos problemas de classificação, tornando o modelo uma base sólida para aplicações em segurança financeira e análise de risco.

# 📊 Relatório Final do Projeto
Detecção de Fraudes em Transações de Criptomoedas com Machine Learning

## 1. Introdução

O crescimento das transações em criptomoedas trouxe novas oportunidades financeiras, mas também aumentou significativamente os riscos de fraudes e atividades maliciosas. Sistemas automatizados capazes de identificar padrões suspeitos são essenciais para reduzir perdas financeiras e aumentar a segurança das transações.

Este projeto tem como objetivo desenvolver um modelo de Machine Learning capaz de identificar transações fraudulentas em redes de criptomoedas, utilizando técnicas de engenharia de atributos, pré-processamento de dados e algoritmos de classificação.

O modelo foi construído utilizando Python, Scikit-Learn e XGBoost, integrando todas as etapas de processamento em um pipeline automatizado de Machine Learning.

## 2. Objetivos do Projeto

#### O projeto possui os seguintes objetivos principais:

Detectar automaticamente transações fraudulentas.

Criar um pipeline de Machine Learning automatizado.

Aplicar boas práticas de pré-processamento de dados.

Construir um modelo escalável que possa ser utilizado em produção.

Demonstrar um fluxo completo de Data Science aplicado a problemas reais.

## 3. Conjunto de Dados

O dataset utilizado contém informações sobre transações realizadas em blockchain, incluindo características como:

número de transações enviadas e recebidas

tempo entre transações

valores mínimos e máximos transferidos

quantidade de contratos criados

saldo total da carteira

interações com tokens ERC20

#### Essas variáveis descrevem o comportamento financeiro de cada endereço na blockchain, permitindo identificar padrões associados a atividades fraudulentas.

## 4. A variável alvo utilizada foi:

## flag

Durante a etapa de engenharia de atributos foram realizadas as seguintes operações:

#### Limpeza dos nomes das colunas

Padronização para evitar inconsistências durante o processamento.

    df.columns = df.columns.str.strip().str.lower()
    
#### Seleção de atributos

Foram removidas variáveis irrelevantes ou redundantes.

    X = df[atributos]
    y = df['flag']

#### Verificação de variabilidade

Foi realizada análise de valores únicos por variável para garantir que os atributos possuam informação relevante para o modelo.

## 5. Pré-processamento dos Dados

O pré-processamento foi realizado utilizando Pipeline e ColumnTransformer do Scikit-Learn, garantindo reprodutibilidade e evitando vazamento de dados.

### As transformações aplicadas foram:

#### Tratamento de valores ausentes

Valores faltantes foram substituídos pela média da variável.

    SimpleImputer(strategy='mean')
    
#### Padronização dos dados

Aplicação do StandardScaler para normalizar a escala das variáveis.

    StandardScaler()
    
Pipeline de Pré-processamento

Imputação → Padronização → Modelo

Isso garante que todas as etapas sejam executadas automaticamente durante o treinamento e previsão.

## 6. Divisão Treino e Teste

O conjunto de dados foi dividido em:

80% treino | 20% teste

Utilizando estratificação para preservar a proporção de fraudes no dataset.

    train_test_split(stratify=y)
    
## 7. Modelo de Machine Learning

O algoritmo escolhido foi o XGBoost (Extreme Gradient Boosting), amplamente utilizado em problemas de classificação devido à sua alta performance e robustez.

* Configuração do modelo:

        XGBClassifier(
            random_state=42,
            eval_metric='auc',
            objective='binary:logistic'
        )

O modelo foi integrado ao pipeline para garantir que todo o fluxo de transformação seja aplicado automaticamente.

## 8. Avaliação do Modelo

A principal métrica utilizada foi AUC (Area Under the ROC Curve).

Essa métrica mede a capacidade do modelo de distinguir entre:

transações legítimas | transações fraudulentas

Valores de AUC próximos de 1 indicam excelente capacidade de classificação.

## 9. Pipeline Final

O pipeline completo inclui:

1️⃣ Pré-processamento

2️⃣ Treinamento do modelo

3️⃣ Predição automática

Isso permite que novos dados sejam processados da mesma forma que os dados de treinamento.

## 10. Uso do Modelo em Produção

Após o treinamento, o modelo pode ser salvo para uso em sistemas reais.

    joblib.dump(pipeline_final, 'modelo_fraude.joblib')

Posteriormente pode ser carregado para realizar previsões em novas transações.

    modelo = joblib.load('modelo_fraude.joblib')
    
## 11. Aplicações do Projeto

**Este tipo de modelo pode ser utilizado em diversos contextos:**

Sistemas antifraude em criptomoedas

Monitoramento de carteiras suspeitas em redes blockchain.

Bancos e fintechs

Detecção de padrões de fraude em transferências financeiras.

Exchanges de criptomoedas

Identificação de contas com comportamento suspeito.

Monitoramento de risco financeiro

Classificação automática de atividades potencialmente fraudulentas.

## 12. Conclusão

Este projeto demonstrou a construção de um sistema de detecção de fraudes utilizando técnicas modernas de Machine Learning e engenharia de dados.

A utilização de pipelines do Scikit-Learn garante um fluxo robusto, reprodutível e pronto para integração em sistemas reais.

A metodologia aplicada pode ser facilmente adaptada para diversos problemas de classificação, tornando o modelo uma base sólida para aplicações em segurança financeira e análise de risco.




# Fim..